In [16]:
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv()

checkpointer = InMemorySaver()
store = InMemoryStore()

@dataclass(frozen=True)
class UserContext:
  user_id: str
  tenant_id: str

def preference_namespace(ctx: UserContext) -> tuple[str, ...]:
  return (
    "platform",
    ctx.tenant_id,
    ctx.user_id
  )

key = "learning_preference"

USER_DATABASE = {
  "school_a": {
    "user_a": {"name": "张三", "course": "langchain开发指南"},
    "user_b": {"name": "李四", "course": "java"},
  },
  "school_b": {
    "user_a": {"name": "王五", "course": "python基础"},
    "user_b": {"name": "丁六", "course": "fastapi构建"},
  }
}

In [17]:
@tool
def get_user_info(runtime: ToolRuntime[UserContext]):
  """获取当前登录用户的信息"""
  ctx = runtime.context
  print(ctx)
  profile = USER_DATABASE.get(ctx.tenant_id, {}).get(ctx.user_id)
  return f"姓名：{profile["name"]}，课程：{profile["course"]}"

@tool
def save_learning_preference(
  preference_name: str,
  preference_value: str,
  runtime: ToolRuntime[UserContext]
) -> str:
  """保存当前用户的长期学习偏好"""

  namespace = preference_namespace(runtime.context)
  print(namespace)
  # 通过namespace和key来读取用户偏好
  item = runtime.store.get(namespace, key)
  preference = dict(item.value) if item else {}

  # 记录本次用户偏好
  preference[preference_name] = preference_value

  # 保存到store中
  runtime.store.put(namespace, key, preference)

  return f"已保存偏好：{preference_name}={preference_value}"
  
@tool
def get_learning_preference(runtime: ToolRuntime[UserContext]) -> str:
  """读取用户的当前学习偏好"""

  namespace = preference_namespace(runtime.context)
  item = runtime.store.get(namespace, key)

  if item is None:
    return "当前用户还没有学习偏好"
  return f"当前学习偏好：{item.value}"


In [18]:
agent = create_agent(
  model="deepseek-flash",
  store=store,
  checkpointer=checkpointer,
  tools=[
    get_user_info,
    save_learning_preference,
    get_learning_preference
  ],
  context_schema=UserContext,
  system_prompt=(
    "你是课程平台助手"
    "涉及用户资料时调用资料工具"
    "涉及长期学习偏好或给用户讲解代码时调用偏好工具"
  )
)

context = UserContext(user_id="user_a", tenant_id="school_b")

result = agent.invoke(
  {"messages": [HumanMessage("记住，我喜欢看代码，再看原理")]},
  config={ "configurable": {"thread_id": "thread_1"} },
  context=context
)

for message in result["messages"]:
  message.pretty_print()

('platform', 'school_b', 'user_a')
================================ Human Message =================================

记住，我喜欢看代码，再看原理
================================== Ai Message ==================================
Tool Calls:
  save_learning_preference (call_00_mHWXR2UXMV9LNeWEVlHM9270)
 Call ID: call_00_mHWXR2UXMV9LNeWEVlHM9270
  Args:
    preference_name: 讲解顺序偏好
    preference_value: 先看代码示例，再看原理讲解（代码优先，原理其次）
================================= Tool Message =================================
Name: save_learning_preference

已保存偏好：讲解顺序偏好=先看代码示例，再看原理讲解（代码优先，原理其次）
================================== Ai Message ==================================

已记住 ✅

**你的偏好**：讲解顺序偏好 = 先看代码示例，再看原理讲解

以后我给你讲任何内容，都会按这个顺序来：

1. **先给可运行的代码**（带上关键注释，能直接跑）
2. **再讲背后的原理**（为什么这样写、底层机制、常见坑）

如果遇到特别抽象的概念，我会先用一段最小代码把它“跑出来”给你看，再展开理论。

要不要现在就拿一个具体知识点试试这个模式？你告诉我主题（比如 Python 装饰器、JS 事件循环、SQL 索引等），我就按“代码 → 原理”的顺序给你讲。


In [19]:
result = agent.invoke(
  {"messages": [HumanMessage("langchain如何实现结构化输出")]},
  config={"configurable": {"thread_id": "thread_2"}},
  context=context
)

for message in result["messages"]:
  message.pretty_print()

UserContext(user_id='user_a', tenant_id='school_b')
================================ Human Message =================================

langchain如何实现结构化输出
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_pi3BA9zbKQSvczUmJ49T1688)
 Call ID: call_00_pi3BA9zbKQSvczUmJ49T1688
  Args:
  get_learning_preference (call_01_A5L6bNQvKh37J6RSlwLy8441)
 Call ID: call_01_A5L6bNQvKh37J6RSlwLy8441
  Args:
================================= Tool Message =================================
Name: get_user_info

姓名：王五，课程：python基础
================================= Tool Message =================================
Name: get_learning_preference

当前学习偏好：{'讲解顺序偏好': '先看代码示例，再看原理讲解（代码优先，原理其次）'}
================================== Ai Message ==================================

王五同学，你好！根据你的学习偏好（**先看代码示例，再看原理讲解**），我先给你可直接运行的代码示例，再讲背后的原理。

---

## 一、先看代码示例

LangChain 实现结构化输出主要有三种方式，从简单到灵活依次是：`Pydantic 模型`、`TypedDict`、`JSON Schema`。最常用的是第一种。

### 方式 1：使用 Pyd